# MS MARCO RARS-v2.2 FP32 held-out seed replication

## tl;dr

This notebook preserves the audited seed-42 Stage-A run and executes only the still-unseen seeds 43 and 44. It re-materializes the two inner bundles with the exact frozen bb9 implementation, requires their registered hashes to match byte for byte, runs both held-out seeds without revealing an interim metric, and applies the frozen replication decision.

This checked-in notebook claims no seed-43 or seed-44 result until it has executed top-to-bottom. The experiment remains development-only and does not access an outer/test outcome or authorize QAT by itself.


## Context & Methods

The training implementation remains pinned to bb9b106e69b9a453756fd800665f701614ce67b3. The post-seed-42 replication protocol and CPU-only aggregator are pinned separately so the frozen trainer is not modified.

The primary replication evidence comes from seeds 43 and 44. Seed 42 is known and is used only as an audited anchor and in the descriptive all-three summary. The machine-readable protocol distinguishes STABLE_GO_TO_QAT, UNSTABLE_NO_QAT, STOP, and INVALID.

### Key assumptions

- Drive still contains the exact MS MARCO 1M source artifacts and the audited seed-42 run.
- A T4/CUDA runtime is selected.
- The runtime matches seed 42 on Python, NumPy, Torch, and CUDA versions.
- Shared qrels are parsed only to reproduce the registered inner bundles. No outer/test outcome enters training, selection, aggregation, or the decision.


In [ ]:
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy==1.26.4', 'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
installed_numpy_version = subprocess.check_output([
    sys.executable, '-c', 'import numpy; print(numpy.__version__)',
], text=True).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version
if 'numpy' in sys.modules:
    assert sys.modules['numpy'].__version__ == '1.26.4', (
        'NumPy was already loaded before the pin took effect. Restart the Colab runtime, then Run all.'
    )

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, math, os, shutil
from datetime import datetime, timezone
from pathlib import Path

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
CONTROL_COMMIT = '00a0dee30767b04b8c650c28d63f4f662ef61517'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
CONTROL_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2_replication_control')
WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
BUNDLES = WORK / 'bundles'
CANDIDATE_CACHE = WORK / 'candidate-cache'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
OUTPUT = DRIVE / 'rars-v2.2-fp32-msmarco' / TRAINING_COMMIT[:12]
SEED42_DIR = OUTPUT / 'seed42-fp32-stage-a'
REPLICATION_ROOT = OUTPUT / 'fp32-replication-v1'
SEED_DIRS = {
    42: SEED42_DIR,
    43: REPLICATION_ROOT / 'seed43-fp32',
    44: REPLICATION_ROOT / 'seed44-fp32',
}
AUDIT_DIR = REPLICATION_ROOT / f'input-audit-{CONTROL_COMMIT[:12]}'
ENVIRONMENT_PATH = REPLICATION_ROOT / 'replication_environment.json'
PIP_FREEZE_PATH = REPLICATION_ROOT / 'pip_freeze.txt'
BATCH_PATH = REPLICATION_ROOT / 'heldout_batch_execution.json'
AGGREGATE_DIR = REPLICATION_ROOT / f'aggregate-{CONTROL_COMMIT[:12]}'
RUNNER_LOGS = REPLICATION_ROOT / 'runner-logs'
BATCH_HISTORY = REPLICATION_ROOT / 'batch-history'
for directory in (REPLICATION_ROOT, AUDIT_DIR, RUNNER_LOGS, BATCH_HISTORY):
    directory.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    temporary.replace(path)

def file_record(path):
    path = Path(path)
    return {'path': str(path), 'bytes': path.stat().st_size, 'sha256': sha256_file(path)}


In [ ]:
def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    dirty = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain'], text=True
    ).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(CONTROL_REPO, CONTROL_COMMIT)

PROTOCOL_PATH = CONTROL_REPO / 'protocols/rars_v2_2_fp32_replication_v1.json'
protocol = json.loads(PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_HELDOUT_SEED_43_AND_44_OUTCOMES'
assert protocol['execution_lineage']['training_source_commit'] == TRAINING_COMMIT
assert protocol['seed_policy']['registered_seeds'] == [42, 43, 44]
assert protocol['seed_policy']['heldout_primary_seeds'] == [43, 44]

subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
    'tests/test_boundary_loss_sidecar.py',
    'tests/test_aggregate_rars_v2_2_fp32_replication.py',
], cwd=CONTROL_REPO, check=True)
print('Exact training commit:', TRAINING_COMMIT)
print('Exact replication-control commit:', CONTROL_COMMIT)
print('Replication protocol SHA-256:', sha256_file(PROTOCOL_PATH))


In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
    SEED42_DIR / 'training_started.json',
    SEED42_DIR / 'training_complete.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 4_000_000_000, 'Need 4 GB local disk'

registered_hashes = protocol['execution_lineage']['source_hashes']
source_files = {
    'bundle_builder_sha256': TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py',
    'bundle_freezer_sha256': TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py',
    'parent_protocol_sha256': TRAIN_REPO / 'protocols/rars_v2_2_boundary_loss_development_v1.json',
    'trainer_sha256': TRAIN_REPO / 'scripts/train_boundary_loss_sidecar_v2_2.py',
    'core_sha256': TRAIN_REPO / 'scripts/rars_v2_2_core.py',
    'aggregator_sha256': CONTROL_REPO / 'scripts/aggregate_rars_v2_2_fp32_replication.py',
    'metric_helper_sha256': CONTROL_REPO / 'scripts/train_boundary_loss_sidecar.py',
}
for key, path in source_files.items():
    assert sha256_file(path) == registered_hashes[key], (key, path)

seed42_anchor = protocol['audited_seed_42']
seed42_complete = json.loads((SEED42_DIR / 'training_complete.json').read_text())
assert seed42_complete['status'] == 'TRAINING_COMPLETE'
assert seed42_complete['run_id'] == seed42_anchor['run_id']
for filename, expected in seed42_anchor['verified_outputs'].items():
    path = SEED42_DIR / filename
    assert path.stat().st_size == expected['bytes'], filename
    assert sha256_file(path) == expected['sha256'], filename
    registered = seed42_complete['outputs'][filename]
    assert registered['bytes'] == expected['bytes']
    assert registered['sha256'] == expected['sha256']
seed42_summary = json.loads((SEED42_DIR / 'training_summary.json').read_text())
assert seed42_summary['selected_epoch'] == seed42_anchor['selected_epoch']
for key, value in seed42_anchor['metrics'].items():
    actual = seed42_summary['selection'][key]
    if isinstance(value, float):
        assert math.isclose(actual, value, rel_tol=0.0, abs_tol=1e-15), key
    else:
        assert actual == value, key
seed42_started = json.loads((SEED42_DIR / 'training_started.json').read_text())

import numpy as np
import torch

current_training_environment = {
    'python': sys.version,
    'numpy': np.__version__,
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'device': 'cuda',
}
assert current_training_environment == seed42_started['environment'], {
    'seed42': seed42_started['environment'],
    'current': current_training_environment,
}
environment_contract = protocol['execution_environment_contract']
assert sys.version_info[:3] == (3, 12, 13), sys.version
assert np.__version__ == environment_contract['numpy_version'], np.__version__
assert torch.__version__ == environment_contract['torch_version'], torch.__version__
assert str(torch.version.cuda) == environment_contract['torch_cuda_version']
assert torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
assert environment_contract['gpu_name_must_contain'] in gpu_name, gpu_name
assert f'{capability[0]}.{capability[1]}' == environment_contract['compute_capability']
assert os.environ.get('CUBLAS_WORKSPACE_CONFIG', 'UNSET') == environment_contract['cublas_workspace_config']
assert torch.backends.cudnn.benchmark is environment_contract['cudnn_benchmark']
torch.use_deterministic_algorithms(True)
assert torch.are_deterministic_algorithms_enabled()
try:
    probe = torch.ones((2, 2), dtype=torch.float32, device='cuda')
    _ = probe @ probe
    torch.cuda.synchronize()
except RuntimeError as error:
    raise AssertionError('Deterministic CUDA matmul preflight failed before held-out training') from error
print('Audited seed 42 verified without rerunning or modifying it.')


## Data

The original seed-42 bundle payloads were ephemeral, so this notebook materializes inner_train and inner_validation once with the exact bb9 builder/freezer and the same absolute paths. The run proceeds only if the two role manifests and split-audit file reproduce the seed-42 hashes exactly.

The inherited builder opens the shared 6,980-query qrels container. Only the two inner roles enter labels and metrics. Outer and clean-test split identities are read only for the disjointness audit; no outer/test relevance or outcome is evaluated.


In [ ]:
builder = [
    sys.executable, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True)
bundle_summary = json.loads((BUNDLES / 'bundle_build_summary.json').read_text())
assert bundle_summary['outer_validation_built'] is False
assert set(bundle_summary['roles']) == {'inner_train', 'inner_validation'}

subprocess.run([
    sys.executable, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True)

lineage = protocol['execution_lineage']
materialization = protocol['data_access_policy']['bundle_materialization']
actual_hashes = {
    'inner_train_manifest': sha256_file(BUNDLES / 'inner_train/v2_2_manifest.json'),
    'inner_validation_manifest': sha256_file(BUNDLES / 'inner_validation/v2_2_manifest.json'),
    'split_audit': sha256_file(BUNDLES / 'v2_2_split_audit.json'),
}
assert actual_hashes['inner_train_manifest'] == materialization['exact_inner_train_manifest_sha256_required']
assert actual_hashes['inner_validation_manifest'] == materialization['exact_inner_validation_manifest_sha256_required']
assert actual_hashes['split_audit'] == materialization['exact_split_audit_sha256_required']
assert actual_hashes['inner_train_manifest'] == lineage['source_hashes']['inner_train_manifest_sha256']
assert actual_hashes['inner_validation_manifest'] == lineage['source_hashes']['inner_validation_manifest_sha256']
assert actual_hashes['split_audit'] == lineage['split_audit_sha256']

for role in ('inner_train', 'inner_validation'):
    manifest = json.loads((BUNDLES / role / 'v2_2_manifest.json').read_text())
    access = manifest['data_access']
    assert access['closed_test_relevance_values_used'] is False
    assert access['closed_test_outcomes_computed'] is False
    assert access['outer_relevance_values_used'] is False
    assert access['outer_outcomes_used'] is False
print(json.dumps({'status': 'EXACT_BUNDLES_REMATERIALIZED', **actual_hashes}, indent=2))


In [ ]:
audit_sources = [
    BUNDLES / 'bundle_build_summary.json',
    BUNDLES / 'v2_2_split_audit.json',
    BUNDLES / 'v2_2_freeze_summary.json',
]
for role in ('inner_train', 'inner_validation'):
    audit_sources.extend([
        BUNDLES / role / 'manifest.json',
        BUNDLES / role / 'query_manifest.json',
        BUNDLES / role / 'v2_2_manifest.json',
    ])
audit_records = []
for source in audit_sources:
    relative = source.relative_to(BUNDLES)
    destination = AUDIT_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    assert sha256_file(destination) == sha256_file(source)
    audit_records.append({
        'relative_path': str(relative),
        'source': file_record(source),
        'persisted_copy': file_record(destination),
    })
write_json(AUDIT_DIR / 'input_audit_manifest.json', {
    'schema_version': 1,
    'protocol_id': protocol['protocol_id'],
    'training_source_commit': TRAINING_COMMIT,
    'control_commit': CONTROL_COMMIT,
    'records': audit_records,
})

import faiss
import numpy as np
import torch

assert sys.version_info[:3] == (3, 12, 13), sys.version
assert np.__version__ == '1.26.4', np.__version__
assert torch.__version__ == '2.11.0+cu128', torch.__version__
assert torch.version.cuda == '12.8', torch.version.cuda
assert torch.cuda.is_available()
torch.use_deterministic_algorithms(True)
assert torch.are_deterministic_algorithms_enabled()

pip_freeze = subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True
)
pip_freeze = '\n'.join(sorted(line for line in pip_freeze.splitlines() if line.strip())) + '\n'
if PIP_FREEZE_PATH.exists():
    assert PIP_FREEZE_PATH.read_text() == pip_freeze
else:
    PIP_FREEZE_PATH.write_text(pip_freeze)

driver = subprocess.check_output([
    'nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'
], text=True).strip().splitlines()[0]
capability = torch.cuda.get_device_capability(0)
environment = {
    'schema_version': 1,
    'protocol_id': protocol['protocol_id'],
    'training_source_commit': TRAINING_COMMIT,
    'control_commit': CONTROL_COMMIT,
    'seed42_run_id': seed42_anchor['run_id'],
    'trainer_sha256': registered_hashes['trainer_sha256'],
    'core_sha256': registered_hashes['core_sha256'],
    'aggregator_sha256': registered_hashes['aggregator_sha256'],
    'metric_helper_sha256': registered_hashes['metric_helper_sha256'],
    'versions': {
        'python_version': '.'.join(map(str, sys.version_info[:3])),
        'numpy_version': np.__version__,
        'torch_version': torch.__version__,
        'torch_cuda_version': str(torch.version.cuda),
        'faiss_version': str(getattr(faiss, '__version__', 'UNKNOWN')),
    },
    'hardware': {
        'device': 'cuda',
        'gpu_name': torch.cuda.get_device_name(0),
        'compute_capability': f'{capability[0]}.{capability[1]}',
        'cudnn_version': str(torch.backends.cudnn.version()),
        'cuda_driver_version': driver,
    },
    'determinism': {
        'deterministic_algorithms_supported_and_enabled': torch.are_deterministic_algorithms_enabled(),
        'cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
        'cudnn_benchmark': bool(torch.backends.cudnn.benchmark),
        'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG', 'UNSET'),
    },
    'pip_freeze_sha256': sha256_file(PIP_FREEZE_PATH),
    'protocol_sha256': sha256_file(PROTOCOL_PATH),
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
if ENVIRONMENT_PATH.exists():
    existing_environment = json.loads(ENVIRONMENT_PATH.read_text())
    comparable_existing = dict(existing_environment)
    comparable_current = dict(environment)
    comparable_existing.pop('created_utc', None)
    comparable_current.pop('created_utc', None)
    assert comparable_existing == comparable_current, 'Existing replication environment differs'
    environment = existing_environment
else:
    write_json(ENVIRONMENT_PATH, environment)
print(json.dumps(environment, indent=2))


## Results

Run seeds 43 and 44 as one sealed batch. Both subprocesses finish before any selection metric is loaded or displayed. Standard output and error are captured in the audit folder. A complete run may be reused only through the frozen trainer's fingerprint-and-hash check; a partial or mismatched directory remains invalid.


In [ ]:
HELDOUT_SEEDS = (43, 44)
assert tuple(protocol['seed_policy']['heldout_primary_seeds']) == HELDOUT_SEEDS
assert len({SEED_DIRS[seed] for seed in HELDOUT_SEEDS}) == len(HELDOUT_SEEDS)

base_trainer = [
    sys.executable, str(TRAIN_REPO / 'scripts/train_boundary_loss_sidecar_v2_2.py'),
    '--bundle-dir', str(BUNDLES / 'inner_train'),
    '--selection-bundle-dir', str(BUNDLES / 'inner_validation'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--source-commit', TRAINING_COMMIT,
    '--rank', '16', '--top-b', '40', '--final-k', '10',
    '--epochs', '10', '--batch-size', '2048', '--score-batch-size', '256',
    '--max-negatives-per-positive', '8', '--promotion-mix', '0.8',
    '--minimum-margin', '0.0001', '--margin-multiplier', '1.0',
    '--learning-rate', '0.0001', '--weight-decay', '0.0001',
    '--correction-l2', '0.001', '--max-correction', '0.05',
    '--max-grad-norm', '5.0',
    '--minimum-gain-over-base', '0.01135',
    '--minimum-gain-over-pca', '0.005',
    '--device', 'cuda', '--reuse-complete',
]

for seed in HELDOUT_SEEDS:
    run_dir = SEED_DIRS[seed]
    if run_dir.exists() and any(run_dir.iterdir()) and not (run_dir / 'training_complete.json').exists():
        raise RuntimeError(
            f'{run_dir} is a preserved partial attempt. Archive and document it before an exogenous same-seed retry.'
        )

batch_attempt_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
batch_records = []
for seed in HELDOUT_SEEDS:
    run_dir = SEED_DIRS[seed]
    preexisting_complete = (run_dir / 'training_complete.json').exists()
    command = base_trainer + ['--output-dir', str(run_dir), '--seed', str(seed)]
    started_utc = datetime.now(timezone.utc).isoformat()
    completed = subprocess.run(
        command, cwd=TRAIN_REPO, text=True, capture_output=True
    )
    finished_utc = datetime.now(timezone.utc).isoformat()
    stdout_path = RUNNER_LOGS / f'seed{seed}_{batch_attempt_id}_stdout.txt'
    stderr_path = RUNNER_LOGS / f'seed{seed}_{batch_attempt_id}_stderr.txt'
    stdout_path.write_text(completed.stdout)
    stderr_path.write_text(completed.stderr)
    batch_records.append({
        'seed': seed,
        'output_dir': str(run_dir),
        'preexisting_complete': preexisting_complete,
        'started_utc': started_utc,
        'finished_utc': finished_utc,
        'return_code': completed.returncode,
        'stdout': file_record(stdout_path),
        'stderr': file_record(stderr_path),
        'training_complete_present': (run_dir / 'training_complete.json').exists(),
    })

batch_payload = {
    'schema_version': 1,
    'protocol_id': protocol['protocol_id'],
    'batch_attempt_id': batch_attempt_id,
    'heldout_seeds': list(HELDOUT_SEEDS),
    'interim_metrics_revealed': False,
    'environment_manifest': file_record(ENVIRONMENT_PATH),
    'runs': batch_records,
}
write_json(BATCH_HISTORY / f'heldout_batch_{batch_attempt_id}.json', batch_payload)
write_json(BATCH_PATH, batch_payload)
print(json.dumps({
    'heldout_batch_finished': True,
    'return_codes': {str(row['seed']): row['return_code'] for row in batch_records},
    'interim_metrics_revealed': False,
}, indent=2))
assert all(row['return_code'] == 0 for row in batch_records), {
    row['seed']: row['return_code'] for row in batch_records
}
assert all(row['training_complete_present'] for row in batch_records)


In [ ]:
def verify_replication_complete(directory):
    complete_path = directory / 'replication_complete.json'
    if not complete_path.exists():
        return None
    complete = json.loads(complete_path.read_text())
    assert complete['status'] == 'REPLICATION_COMPLETE'
    for filename, record in complete['outputs'].items():
        path = directory / filename
        assert path.stat().st_size == record['bytes'], filename
        assert sha256_file(path) == record['sha256'], filename
    return json.loads((directory / 'replication_summary.json').read_text())

replication_summary = verify_replication_complete(AGGREGATE_DIR)
if replication_summary is None:
    aggregate_command = [
        sys.executable,
        str(CONTROL_REPO / 'scripts/aggregate_rars_v2_2_fp32_replication.py'),
        '--train-bundle-dir', str(BUNDLES / 'inner_train'),
        '--selection-bundle-dir', str(BUNDLES / 'inner_validation'),
        '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
        '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
        '--replication-protocol', str(PROTOCOL_PATH),
        '--environment-manifest', str(ENVIRONMENT_PATH),
        '--seed-42-dir', str(SEED_DIRS[42]),
        '--seed-43-dir', str(SEED_DIRS[43]),
        '--seed-44-dir', str(SEED_DIRS[44]),
        '--output-dir', str(AGGREGATE_DIR),
    ]
    completed = subprocess.run(
        aggregate_command, cwd=CONTROL_REPO, text=True, capture_output=True
    )
    if completed.returncode != 0:
        print(completed.stderr)
        raise subprocess.CalledProcessError(
            completed.returncode, aggregate_command,
            output=completed.stdout, stderr=completed.stderr,
        )
    replication_summary = json.loads(
        (AGGREGATE_DIR / 'replication_summary.json').read_text()
    )
runner_artifacts = [
    BATCH_PATH, ENVIRONMENT_PATH, PIP_FREEZE_PATH,
    AUDIT_DIR / 'input_audit_manifest.json',
    AGGREGATE_DIR / 'replication_complete.json',
    SEED_DIRS[43] / 'training_complete.json',
    SEED_DIRS[44] / 'training_complete.json',
]
runner_artifacts.extend(sorted(RUNNER_LOGS.glob('seed*_std*.txt')))
runner_artifacts.extend(sorted(BATCH_HISTORY.glob('heldout_batch_*.json')))
write_json(REPLICATION_ROOT / 'replication_runner_manifest.json', {
    'schema_version': 1,
    'protocol_id': protocol['protocol_id'],
    'training_source_commit': TRAINING_COMMIT,
    'control_commit': CONTROL_COMMIT,
    'artifacts': [file_record(path) for path in runner_artifacts],
})
print('Replication aggregation and artifact verification complete.')


In [ ]:
decision = json.loads((AGGREGATE_DIR / 'replication_decision.json').read_text())
bootstrap = json.loads((AGGREGATE_DIR / 'paired_bootstrap.json').read_text())
report = {
    'decision': decision['decision'],
    'qat_protocol_definition_authorized': decision['qat_protocol_definition_authorized'],
    'comparators': replication_summary['comparators'],
    'per_seed': replication_summary['per_seed'],
    'heldout_replication': replication_summary['groups']['heldout_replication_seeds'],
    'all_three_seeds': replication_summary['groups']['all_seeds'],
    'heldout_bootstrap': {
        'vs_base': bootstrap['contrasts']['heldout_mean_minus_base'],
        'vs_pca_fp32': bootstrap['contrasts']['heldout_mean_minus_pca_fp32'],
    },
    'conditions': decision['conditions'],
    'aggregate_dir': str(AGGREGATE_DIR),
}
print(json.dumps(report, indent=2, allow_nan=False))


## Takeaways

Interpret only the final decision artifact:

- STABLE_GO_TO_QAT authorizes writing a separate frozen QAT protocol. It does not authorize immediate QAT tuning, outer/test evaluation, or a deployment claim.
- UNSTABLE_NO_QAT means the held-out mean effect passed but robustness failed.
- STOP means a held-out mean effect-size gate failed.
- INVALID means provenance or execution could not be audited and is not a scientific result.

Do not add a new seed, drop a failed seed, alter a threshold, or inspect an external outcome under this protocol. Return the final report plus the replication_complete.json and input-audit folder for independent review.
